# BERT Transfer Learning Practice

- KLUE-YNAT 뉴스 제목 주제 분류 실습
- 사전학습된 한국어 BERT 모델을 가져와 새로운 분류 데이터셋에 맞게 fine-tuning한다.

**목표**
1. Hugging Face `datasets`로 KLUE-YNAT 데이터셋을 불러올 수 있다.
2. 뉴스 제목과 라벨 구조를 확인하고 다중 클래스 분류 문제로 정리할 수 있다.
3. `AutoTokenizer`를 사용해 BERT 입력 형태로 텍스트를 변환할 수 있다.
4. `AutoModelForSequenceClassification`으로 분류 모델을 구성할 수 있다.
5. 사전학습 모델을 뉴스 주제 분류 데이터셋에 맞게 전이학습할 수 있다.
6. 평가 결과와 직접 입력한 문장의 예측 결과를 해석할 수 있다.

## 0. 실습 환경 준비

필요한 라이브러리를 불러온다. `datasets`, `transformers`, `accelerate`가 설치되어 있지 않다면 설치 코드를 먼저 실행한다.

In [1]:
# 필요한 경우 아래 설치 코드를 먼저 실행한다.
%pip install datasets transformers accelerate -q

Note: you may need to restart the kernel to use updated packages.


In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader

from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from transformers import get_linear_schedule_with_warmup

from sklearn.metrics import accuracy_score, classification_report
from tqdm.auto import tqdm

plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False

## 1. KLUE-YNAT 데이터셋 불러오기

1. `load_dataset()`을 사용해 `klue`, `ynat` 데이터셋을 불러오기
2. train 데이터와 validation 데이터를 각각 확인하기
3. 데이터셋의 column 이름 확인하기
4. train 데이터의 첫 번째 샘플 확인하기

In [3]:
dataset = load_dataset('klue', 'ynat')

dataset

README.md: 0.00B [00:00, ?B/s]

c:\Users\Playdata\AppData\Local\miniforge3\envs\dl_nlp_env\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Playdata\.cache\huggingface\hub\datasets--klue. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


ynat/train-00000-of-00001.parquet:   0%|          | 0.00/4.17M [00:00<?, ?B/s]

ynat/validation-00000-of-00001.parquet:   0%|          | 0.00/847k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/45678 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/9107 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['guid', 'title', 'label', 'url', 'date'],
        num_rows: 45678
    })
    validation: Dataset({
        features: ['guid', 'title', 'label', 'url', 'date'],
        num_rows: 9107
    })
})

In [4]:
print(dataset.keys())
print(dataset['train'].column_names)
print(dataset['train'][0])

dict_keys(['train', 'validation'])
['guid', 'title', 'label', 'url', 'date']
{'guid': 'ynat-v1_train_00000', 'title': '유튜브 내달 2일까지 크리에이터 지원 공간 운영', 'label': 3, 'url': 'https://news.naver.com/main/read.nhn?mode=LS2D&mid=shm&sid1=105&sid2=227&oid=001&aid=0008508947', 'date': '2016.06.30. 오전 10:36'}


## 2. 데이터프레임 변환 및 기본 구조 확인하기

1. train 데이터를 `train_df`로 변환하기
2. validation 데이터를 `val_df`로 변환하기
3. `title`, `label` 컬럼만 사용하기
4. 데이터 크기와 결측치 개수 확인하기
5. 라벨 이름 목록 확인하기

## 3. 라벨 분포 확인하기

1. 라벨 id와 라벨 이름을 연결하는 `id2label` 딕셔너리 만들기
2. 라벨 이름과 라벨 id를 연결하는 `label2id` 딕셔너리 만들기
3. train 데이터의 라벨 분포 확인하기
4. 라벨 분포를 막대그래프로 시각화하기

## 4. 실습용 데이터 샘플링하기

전체 데이터를 모두 사용하면 학습 시간이 오래 걸릴 수 있다. 실습에서는 일부 데이터만 사용한다.

1. train 데이터에서 3000개 샘플링하기
2. validation 데이터에서 1000개 샘플링하기
3. `random_state=42`로 설정하기
4. 샘플링된 데이터 크기 확인하기

## 5. BERT Tokenizer 준비하기

1. 모델 이름을 `klue/bert-base`로 설정하기
2. `AutoTokenizer.from_pretrained()`로 tokenizer 불러오기
3. 뉴스 제목 하나를 토큰화하기
4. 토큰과 토큰 id를 함께 확인하기

## 6. 데이터 토큰화하기

1. train 제목과 validation 제목을 리스트로 변환하기
2. 라벨을 리스트로 변환하기
3. `padding=True`, `truncation=True`, `max_length=64`를 적용해 토큰화하기
4. 토큰화 결과의 key 확인하기
5. `input_ids`, `attention_mask`의 shape 확인하기

## 7. 토큰화 결과 점검하기

1. 하나의 뉴스 제목을 선택하기
2. 원문 제목 확인하기
3. `input_ids`를 다시 토큰으로 변환하기
4. `attention_mask` 확인하기
5. 라벨 id와 라벨 이름 확인하기

## 8. TensorDataset과 DataLoader 만들기

1. 라벨 리스트를 tensor로 변환하기
2. `TensorDataset`으로 train dataset 만들기
3. `TensorDataset`으로 validation dataset 만들기
4. `DataLoader`를 생성하기
5. 첫 번째 batch의 shape 확인하기

## 9. BERT 분류 모델 준비하기

1. `AutoModelForSequenceClassification`으로 모델 불러오기
2. `num_labels`를 라벨 개수로 설정하기
3. `id2label`, `label2id`를 모델에 전달하기
4. GPU 사용 가능 여부를 확인하고 device 설정하기
5. 모델을 device로 이동하기

## 10. Optimizer와 Scheduler 설정하기

1. AdamW optimizer를 생성하기
2. learning rate는 `2e-5`로 설정하기
3. epoch는 1로 설정하기
4. 전체 학습 step 수 계산하기
5. linear scheduler 생성하기

## 11. 모델 학습하기

1. 모델을 train mode로 설정하기
2. batch 데이터를 device로 이동하기
3. 모델에 `input_ids`, `attention_mask`, `labels` 전달하기
4. loss를 계산하고 역전파하기
5. gradient clipping 적용하기
6. optimizer와 scheduler를 업데이트하기
7. epoch별 평균 loss를 출력하기

## 12. 검증 데이터 평가하기

1. 모델을 eval mode로 설정하기
2. validation loader를 순회하며 예측값 구하기
3. `argmax`로 예측 라벨 구하기
4. accuracy 출력하기
5. classification report 출력하기

## 13. 직접 입력한 뉴스 제목 예측하기

1. 새 뉴스 제목 리스트 만들기
2. 예측 함수를 작성하기
3. 각 문장에 대해 예측 라벨과 확률 출력하기
4. 예측 결과가 타당한지 확인하기

## 14. 모델과 토크나이저 저장 및 로드하기

1. fine-tuning한 모델 저장하기
2. tokenizer 저장하기
3. 저장된 모델과 tokenizer 다시 불러오기
4. 불러온 모델로 새 문장 예측하기